# 03_cleaning: types, missing values and the clean checkpoint
Input: `data/raw/itsm_raw.csv` (from `02_ingest_and_profile`). Output: `data/processed/itsm_clean.pkl`. Aggregates only.

Steps: placeholders to nulls, presence flags, type conversion, data-quality checks and label fixes, the Handle_Time_hrs check, elapsed time, null treatment per column, and two dropped columns. Scaling, SMOTE and outlier capping are not used: the model features are categorical (one-hot encoded) and the skewed durations are post-event (`04_eda`, section 5).

## 1. Load the raw snapshot

In [1]:
import os
import numpy as np
import pandas as pd

df = pd.read_csv("../data/raw/itsm_raw.csv", dtype=str, keep_default_na=False)
print(df.shape)

(46606, 25)


<details>
<summary><b>Code notes, line by line</b></summary>

- `import os / numpy / pandas` : File paths, numeric arrays and data frames.
- `pd.read_csv(..., dtype=str, keep_default_na=False)` : Reads the raw snapshot with every value as text and blanks kept as empty strings, so nothing is converted before we decide how.
- `print(df.shape)` : Checks 46,606 rows and 25 columns arrived.

</details>

## 2. Column groups (src/columns.py)

In [2]:
%%writefile ../src/columns.py
id_cols       = ["Incident_ID", "KB_number", "Related_Interaction", "Related_Change"]
cat_cols      = ["CI_Name", "CI_Cat", "CI_Subcat", "WBS", "Category", "Closure_Code", "Status", "Alert_Status"]
ordinal_cols  = ["Impact", "Urgency", "Priority"]
count_cols    = ["No_of_Reassignments", "No_of_Related_Interactions", "No_of_Related_Incidents", "No_of_Related_Changes"]
time_cols     = ["Open_Time", "Reopen_Time", "Resolved_Time", "Close_Time"]
duration_cols = ["Handle_Time_hrs"]
drop_cols     = ["number_cnt"]



Overwriting ../src/columns.py


<details>
<summary><b>Code notes, line by line</b></summary>

- `%%writefile ../src/columns.py` : Saves the column groups as a module that other notebooks can import.
- `id_cols / cat_cols / ordinal_cols / count_cols / time_cols / duration_cols / drop_cols` : Each list groups the columns that are cleaned the same way: identifiers, categories, the 1-to-5 scales, counts, timestamps, durations, and columns to drop.

</details>

## 3. Check that all 25 columns are grouped

In [3]:
import sys, importlib
importlib.invalidate_caches()
sys.path.append("../src")
from columns import id_cols, cat_cols, ordinal_cols, count_cols, time_cols, duration_cols, drop_cols

grouped = id_cols + cat_cols + ordinal_cols + count_cols + time_cols + duration_cols + drop_cols
print(len(grouped), set(df.columns) - set(grouped))

25 set()


<details>
<summary><b>Code notes, line by line</b></summary>

- `importlib.invalidate_caches(); sys.path.append("../src")` : Lets Python find the module just written.
- `from columns import ...` : Imports the seven groups.
- `grouped = id_cols + ... + drop_cols` : All grouped column names in one list.
- `print(len(grouped), set(df.columns) - set(grouped))` : Should print 25 and an empty set: every column belongs to exactly one group.

</details>

## 4. Placeholders to nulls

In [4]:
placeholders = ["", "?", "NA", "N/A", "NaN", "nan", "null", "NULL", "None", "--", "-", "NS"]
for c in df.columns:
    df[c] = df[c].str.strip().replace(placeholders, np.nan)

print("Table 4. Nulls per column after placeholders are converted")
df.isna().sum()[lambda s: s > 0].sort_values(ascending=False)

Table 4. Nulls per column after placeholders are converted


No_of_Related_Changes         46046
Related_Change                46046
No_of_Related_Incidents       45384
Reopen_Time                   44322
Resolved_Time                  1780
Impact                         1380
Priority                       1380
Closure_Code                    460
No_of_Related_Interactions      114
CI_Cat                          111
CI_Subcat                       111
No_of_Reassignments               1
Handle_Time_hrs                   1
dtype: int64

<details>
<summary><b>Code notes, line by line</b></summary>

- `placeholders = [...]` : The missing-value markers found in 02 ("Unknown" is not included: it is a real CI category value).
- `df[c].str.strip().replace(placeholders, np.nan)` : For every column: trims spaces and turns placeholders into true nulls.
- `df.isna().sum()[lambda s: s > 0].sort_values(...)` : Shows only the columns that now have nulls, largest first.

</details>

## 5. Presence flags, created before type conversion

In [5]:
df["Has_Reopen_Time"] = df["Reopen_Time"].notna().astype(int)
df["Has_Related_Change"] = df["Related_Change"].notna().astype(int)
print("Table 5. Tickets with each presence flag")
print(df[["Has_Reopen_Time", "Has_Related_Change"]].sum())

Table 5. Tickets with each presence flag
Has_Reopen_Time       2284
Has_Related_Change     560
dtype: int64


<details>
<summary><b>Code notes, line by line</b></summary>

- `df["Has_Reopen_Time"] = df["Reopen_Time"].notna().astype(int)` : 1 if the ticket was reopened, else 0; made before conversion so a failed date parse cannot change it.
- `df["Has_Related_Change"] = df["Related_Change"].notna().astype(int)` : 1 if a change is linked: this becomes the Track 4 target.
- `print(df[[...]].sum())` : Number of tickets with each flag.

</details>

## 6. Type conversion, with a self-check

In [6]:
num_cols = ordinal_cols + count_cols + duration_cols
before = df[num_cols].isna().sum()

df["Urgency"] = df["Urgency"].replace({"5 - Very Low": "5"})

for c in ordinal_cols + count_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

df["Handle_Time_hrs"] = pd.to_numeric(df["Handle_Time_hrs"].str.replace(",", ".", regex=False), errors="coerce")

for c in time_cols:
    df[c] = pd.to_datetime(df[c], format="%d-%m-%Y %H:%M", errors="coerce")

print("Table 6a. New nulls created by conversion (empty means none)")
print((df[num_cols].isna().sum() - before)[lambda s: s != 0])
print("Table 6b. Nulls in the four date columns after parsing")
print(df[time_cols].isna().sum())
print("Table 6c. Data types after conversion")
df.dtypes

Table 6a. New nulls created by conversion (empty means none)
Handle_Time_hrs    31252
dtype: int64
Table 6b. Nulls in the four date columns after parsing
Open_Time            0
Reopen_Time      44322
Resolved_Time     1780
Close_Time           0
dtype: int64
Table 6c. Data types after conversion


CI_Name                                  str
CI_Cat                                   str
CI_Subcat                                str
WBS                                      str
Incident_ID                              str
Status                                   str
Impact                               float64
Urgency                                int64
Priority                             float64
number_cnt                               str
Category                                 str
KB_number                                str
Alert_Status                             str
No_of_Reassignments                  float64
Open_Time                     datetime64[us]
Reopen_Time                   datetime64[us]
Resolved_Time                 datetime64[us]
Close_Time                    datetime64[us]
Handle_Time_hrs                      float64
Closure_Code                             str
No_of_Related_Interactions           float64
Related_Interaction                      str
No_of_Rela

<details>
<summary><b>Code notes, line by line</b></summary>

- `num_cols = ordinal_cols + count_cols + duration_cols` : Columns that should become numbers.
- `before = df[num_cols].isna().sum()` : Null counts before conversion, to detect values lost by the conversion.
- `df["Urgency"].replace({"5 - Very Low": "5"})` : Fixes the one text label so Urgency converts cleanly.
- `pd.to_numeric(df[c], errors="coerce")` : Converts the scales and counts to numbers; unconvertible values become NaN instead of raising an error.
- `df["Handle_Time_hrs"].str.replace(",", ".") then to_numeric` : Treats the comma as a decimal point and converts.
- `pd.to_datetime(df[c], format="%d-%m-%Y %H:%M", errors="coerce")` : Parses the four timestamps with the exact day-month-year format found in 02.
- `(df[num_cols].isna().sum() - before)[lambda s: s != 0]` : Self-check: new nulls created by the conversion (only Handle_Time_hrs, explained in section 8).
- `df[time_cols].isna().sum(); df.dtypes` : Null counts of the dates (they match the blanks, so the format is right) and the final type of each column.

</details>

**Result.**
* **Null counts (Table 4):** Closure_Code has 460 nulls, as predicted. CI_Cat and CI_Subcat show 111 each, which means those are genuine blanks, not the word "Unknown".
* **Flags (Table 5):** Has_Reopen_Time is 2,284 and Has_Related_Change is 560.
* **Datetimes (Table 6b):** the null counts are 0, 44,322, 1,780 and 0, equal to the blanks, so the dd-mm-yyyy hh:mm format is correct for all four columns; they are now `datetime64` (Table 6c).
* **Other conversions (Table 6a):** Impact, Urgency, Priority and the count columns converted with no new nulls; only Handle_Time_hrs lost 31,252 values, explained in section 8.

## 7. Data-quality checks: dates in order, and values missing together
Timestamps should never run backwards, and blanks that always occur together point to a recording rule rather than random errors.

In [7]:
dq_time = pd.Series({
    "Close_Time before Open_Time": int((df["Close_Time"] < df["Open_Time"]).sum()),
    "Resolved_Time before Open_Time": int((df["Resolved_Time"] < df["Open_Time"]).sum()),
    "Reopen_Time before Open_Time": int((df["Reopen_Time"] < df["Open_Time"]).sum()),
    "Resolved_Time after Close_Time": int((df["Resolved_Time"] > df["Close_Time"]).sum()),
    "Reopen_Time after Close_Time": int((df["Reopen_Time"] > df["Close_Time"]).sum()),
}, name="Tickets").to_frame()
dq_time.index.name = "Check"

pairs = [("Related_Change", "No_of_Related_Changes"), ("CI_Cat", "CI_Subcat"),
         ("Priority", "Impact"), ("Closure_Code", "Resolved_Time")]
dq_missing = pd.DataFrame([{"Column A": a, "Column B": b,
                            "A missing": int(df[a].isna().sum()), "B missing": int(df[b].isna().sum()),
                            "Both missing": int((df[a].isna() & df[b].isna()).sum())}
                           for a, b in pairs]).set_index(["Column A", "Column B"])

no_label = df["Priority"].isna() & df["Impact"].isna()
print("Table 7a. Timestamps out of order")
display(dq_time)
print("Table 7b. Columns missing together")
display(dq_missing)
print("Urgency recorded on the", int(no_label.sum()), "tickets with no Priority and no Impact:",
      int(df.loc[no_label, "Urgency"].notna().sum()))
print("Status of tickets with no Resolved_Time:", df.loc[df["Resolved_Time"].isna(), "Status"].value_counts().to_dict())

Table 7a. Timestamps out of order


,Tickets
Check,
Close_Time before Open_Time,0
Resolved_Time before Open_Time,0
Reopen_Time before Open_Time,0
Resolved_Time after Close_Time,0
Reopen_Time after Close_Time,0


Table 7b. Columns missing together


,,A missing,B missing,Both missing
Column A,Column B,,,
Related_Change,No_of_Related_Changes,46046,46046,46046
CI_Cat,CI_Subcat,111,111,111
Priority,Impact,1380,1380,1380
Closure_Code,Resolved_Time,460,1780,460


Urgency recorded on the 1380 tickets with no Priority and no Impact: 1380
Status of tickets with no Resolved_Time: {'Closed': 1780}


<details>
<summary><b>Code notes, line by line</b></summary>

- `dq_time = pd.Series({...})` : Five order checks, each counting tickets where one timestamp falls before another it should follow (Table 7a).
- `(df["Close_Time"] < df["Open_Time"]).sum()` : Example: tickets closed before they were opened. Comparisons with a missing date count as False.
- `pairs = [("Related_Change", "No_of_Related_Changes"), ...]` : Column pairs suspected of being blank together.
- `dq_missing = pd.DataFrame([{...} for a, b in pairs])` : For each pair: blanks in A, blanks in B, and blanks in both (Table 7b). Equal numbers mean a recording rule.
- `no_label = df["Priority"].isna() & df["Impact"].isna()` : Tickets with neither Priority nor Impact.
- `df.loc[no_label, "Urgency"].notna().sum()` : Whether Urgency is still recorded on them.
- `df.loc[df["Resolved_Time"].isna(), "Status"].value_counts()` : Status of tickets without a resolved time: all Closed.

</details>

**Result.** No timestamp is out of order (Table 7a). Blanks come in fixed pairs (Table 7b): Related_Change and No_of_Related_Changes are blank on the same 46,046 tickets (no linked change); CI_Cat and CI_Subcat on the same 111; Priority and Impact on the same 1,380, all of which still have an Urgency; and all 460 tickets with no Closure_Code are among the 1,780 with no Resolved_Time, every one of them Closed. The blanks follow recording rules rather than random errors, so each pair is treated together in section 10 and neither column of a pair is filled from the other.

### 7b. Duplicate and non-English labels
Labels that differ only in capitals or spacing split one category into two. Each is merged into its most common spelling, and the Dutch Closure_Code "Overig" ("Other") into "Other".

In [8]:
label_cols = ["CI_Cat", "CI_Subcat", "Category", "Closure_Code"]
norm = lambda s: s.str.casefold().str.replace(r"\s+", " ", regex=True)

rows = []
for c in label_cols:
    v = df[c].dropna()
    variants = v.groupby(norm(v)).agg(lambda s: sorted(s.unique()))
    for labels in variants[variants.map(len) > 1]:
        rows.append({"Column": c, "Variants": " / ".join(labels), "Tickets": int(v.isin(labels).sum())})
print("Table 7c. Labels that differ only in capitals or spacing")
display(pd.DataFrame(rows).set_index("Column") if rows else "none found")
print("Non-English Closure_Code labels:",
      {k: int((df["Closure_Code"] == k).sum()) for k in ["Overig", "Kwaliteit van de output"]})

# Fixes: one spelling per variant group (the most common one); Dutch "Overig" means "Other"
for c in label_cols:
    v = df[c].dropna()
    key = norm(v)
    canon = v.groupby(key).agg(lambda s: s.value_counts().index[0])
    df.loc[v.index, c] = key.map(canon)
df["Closure_Code"] = df["Closure_Code"].replace({"Overig": "Other"})
print("CI_Subcat labels after the fix:", df["CI_Subcat"].nunique(), "| Overig left:", int((df["Closure_Code"] == "Overig").sum()))

Table 7c. Labels that differ only in capitals or spacing


,Variants,Tickets
Column,,
CI_Subcat,IPtelephony / Iptelephony,3


Non-English Closure_Code labels: {'Overig': 1, 'Kwaliteit van de output': 10}
CI_Subcat labels after the fix: 63 | Overig left: 0


<details>
<summary><b>Code notes, line by line</b></summary>

- `label_cols = [...]` : The text label columns to check.
- `norm = lambda s: s.str.casefold().str.replace(r"\s+", " ", regex=True)` : A normalised key: lower case and single spaces, so 'IPtelephony' and 'Iptelephony' get the same key.
- `variants = v.groupby(norm(v)).agg(lambda s: sorted(s.unique()))` : For each key, the different spellings that map to it.
- `variants[variants.map(len) > 1]` : Keeps only keys with two or more spellings and adds them to Table 7c with their ticket count.
- `{k: int((df["Closure_Code"] == k).sum()) for k in ["Overig", ...]}` : Counts the two Dutch Closure_Code labels.
- `canon = v.groupby(key).agg(lambda s: s.value_counts().index[0])` : The fix: for each key, the most common spelling becomes the single label.
- `df.loc[v.index, c] = key.map(canon)` : Rewrites every non-blank value with its chosen spelling.
- `df["Closure_Code"].replace({"Overig": "Other"})` : Translates the Dutch 'Overig' into the existing 'Other'.
- `print(... nunique() ...)` : Checks the fix: fewer CI_Subcat labels and no 'Overig' left.

</details>

**Result.** One case variant (Table 7c): CI_Subcat "IPtelephony" / "Iptelephony" (3 tickets), merged into one spelling, so CI_Subcat now has 63 labels (last printed line). Closure_Code has two Dutch labels: "Overig" (1 ticket) means "Other" and is merged into it; "Kwaliteit van de output" ("quality of the output", 10 tickets) has no English equivalent and is kept. All of these are rare and pooled by the encoder, so no model result changes. Labels that are similar but not identical are reviewed in `04_eda`, section 14.

## 8. Handle_Time_hrs: why two thirds fail to convert

In [9]:
raw = pd.read_csv("../data/raw/itsm_raw.csv", dtype=str, keep_default_na=False, usecols=["Handle_Time_hrs"])
h = raw["Handle_Time_hrs"].str.strip()
conv = pd.to_numeric(h.str.replace(",", ".", regex=False), errors="coerce")
bad = h[conv.isna() & (h != "")]

print("Table 8. Handle_Time_hrs values that fail to convert | failing values:", len(bad))
print(bad.str.replace(r"\d", "9", regex=True).value_counts().head(10))
print("Non-digit characters:", sorted({ch for v in bad for ch in v if not ch.isdigit()}))

Table 8. Handle_Time_hrs values that fail to convert | failing values: 31252
Handle_Time_hrs
9,99,99,99,999    30302
9,99,999            849
99,99,999            96
9,99,99,999           5
Name: count, dtype: int64
Non-digit characters: [',']


<details>
<summary><b>Code notes, line by line</b></summary>

- `raw = pd.read_csv(..., usecols=["Handle_Time_hrs"])` : Re-reads only this column in its original text form.
- `conv = pd.to_numeric(h.str.replace(",", "."), errors="coerce")` : Repeats the conversion from section 6.
- `bad = h[conv.isna() & (h != "")]` : Values that are not blank but still failed.
- `bad.str.replace(r"\d", "9", regex=True).value_counts()` : Replaces every digit with 9 to show the shapes of the failing values without showing the values.
- `sorted({ch for v in bad for ch in v if not ch.isdigit()})` : All non-digit characters in them: only the comma, so they are digit-grouped numbers.

</details>

**Result.** 31,252 values, about two thirds, fail (Table 8). They take only four shapes, such as `9,99,99,99,999`, and the comma is the only non-digit character: they are digit-grouped integers with the decimal place lost, so they cannot be read as hours.

**Decision.** Keep the parsed values, leave the rest null, and exclude the column from all features (it is only known after handling); Elapsed_hrs (section 9) is used for descriptive work instead. Assumption: Handle_Time_hrs is agent work time (effort hours), possibly summed across agents, which is why some values exceed the open-to-close span. Evidence and tests: `data_issues_log.md`.

## 9. Elapsed hours, open to close

In [10]:
df["Elapsed_hrs"] = (df["Close_Time"] - df["Open_Time"]).dt.total_seconds() / 3600
print("Table 9. Elapsed_hrs summary (hours, open to close)")
print(df["Elapsed_hrs"].describe().round(1))

Table 9. Elapsed_hrs summary (hours, open to close)
count    46606.0
mean       123.1
std        452.8
min          0.0
25%          1.3
50%         18.4
75%         95.1
max      15722.6
Name: Elapsed_hrs, dtype: float64


<details>
<summary><b>Code notes, line by line</b></summary>

- `(df["Close_Time"] - df["Open_Time"]).dt.total_seconds() / 3600` : Hours from open to close for every ticket (both dates are always present).
- `df["Elapsed_hrs"].describe().round(1)` : Summary: the mean far above the median shows a long right tail.

</details>

**Result.** Elapsed_hrs is complete for all 46,606 tickets and heavily right-skewed: median 18.4 hours, mean 123.1, three quarters under 95.1 hours, maximum 15,722.6 (Table 9). It is post-event, so it is used only for descriptive work (`04_eda`, `09_track3_triage`), never as a feature.

## 10. Null treatment per column

In [11]:
for c in ["No_of_Related_Incidents", "No_of_Related_Changes"]:
    df[c] = df[c].fillna(0)

df["Closure_Code"] = df["Closure_Code"].fillna("Not Recorded")
df["CI_Cat"] = df["CI_Cat"].fillna("Unknown")
df["CI_Subcat"] = df["CI_Subcat"].fillna("Unknown")

<details>
<summary><b>Code notes, line by line</b></summary>

- `for c in ["No_of_Related_Incidents", "No_of_Related_Changes"]: df[c].fillna(0)` : A blank related count means none, so it becomes 0.
- `df["Closure_Code"].fillna("Not Recorded")` : Keeps the 460 blanks as an explicit category instead of guessing a code.
- `df["CI_Cat"].fillna("Unknown"); df["CI_Subcat"].fillna("Unknown")` : The 111 tickets with no asset type get the label 'Unknown' in both columns.

</details>

Blank related counts mean "none" (no zero appears among the filled values), so they become 0. Blank Closure_Code becomes "Not Recorded", and blank CI_Cat and CI_Subcat become "Unknown" (the same 111 tickets). Resolved_Time, Reopen_Time, Priority and Impact stay missing and are not imputed: unlabelled tickets are left out of Track 1.

## 11. Drop Alert_Status and number_cnt, and save the checkpoint
Alert_Status has one value ("closed") and number_cnt holds uniform random values, all unique, that are not in the project brief (evidence: `02_ingest_and_profile`, sections 5 and 7).

In [12]:
df = df.drop(columns=["number_cnt", "Alert_Status"])
os.makedirs("../data/processed", exist_ok=True)
df.to_pickle("../data/processed/itsm_clean.pkl")
print("Table 11. Clean checkpoint: shape and the nulls that remain")
print(df.shape)
print(df.isna().sum()[lambda s: s > 0].sort_values(ascending=False))

Table 11. Clean checkpoint: shape and the nulls that remain
(46606, 26)
Related_Change                46046
Reopen_Time                   44322
Handle_Time_hrs               31253
Resolved_Time                  1780
Impact                         1380
Priority                       1380
No_of_Related_Interactions      114
No_of_Reassignments               1
dtype: int64


<details>
<summary><b>Code notes, line by line</b></summary>

- `df.drop(columns=["number_cnt", "Alert_Status"])` : Removes the random-number column and the single-value column.
- `os.makedirs("../data/processed", exist_ok=True)` : Creates the output folder if needed.
- `df.to_pickle("../data/processed/itsm_clean.pkl")` : Saves the clean checkpoint; a pickle keeps the data types (dates stay dates).
- `print(df.shape); print(df.isna().sum()[...])` : Final size and the nulls that remain on purpose.

</details>

**Result.** The clean checkpoint has 46,606 rows and 26 columns (Table 11). The nulls left are intentional: Related_Change (46,046, no linked change), Reopen_Time (44,322, never reopened), Handle_Time_hrs (31,253 unreadable, section 8), Resolved_Time (1,780), Priority and Impact (1,380 unlabelled tickets), No_of_Related_Interactions (114) and No_of_Reassignments (1).

## 12. Findings
1. **Placeholders.** Blanks and placeholders (`NA`, `NS`, `?` and similar) were turned into true nulls (Table 4); "Unknown" was kept as a real CI category value.
2. **Presence flags.** Has_Reopen_Time (2,284 tickets) and Has_Related_Change (560, the Track 4 target) were created before type conversion (Table 5).
3. **Types.** Impact, Urgency, Priority and the count columns converted with no new nulls; the four dates parse as dd-mm-yyyy hh:mm, with nulls equal to the blanks (Tables 6a to 6c). The one `5 - Very Low` Urgency label was recoded to 5.
4. **Data quality.** No timestamp is out of order (Table 7a), and blanks occur in fixed pairs (Table 7b), so they follow recording rules rather than random errors.
5. **Label fixes.** One case duplicate in CI_Subcat (3 tickets) was merged, and the Dutch Closure_Code "Overig" (1 ticket) was recoded to "Other" (Table 7c).
6. **Handle_Time_hrs.** 31,252 values are digit-grouped integers with the decimal place lost (Table 8); they are left null, and the column is excluded from every feature set (post-event).
7. **Elapsed_hrs.** Hours from open to close were derived for every ticket (median 18.4, mean 123.1; Table 9) for descriptive use only.
8. **Nulls.** Related counts were filled with 0, Closure_Code with "Not Recorded" and CI_Cat and CI_Subcat with "Unknown"; Resolved_Time, Reopen_Time, Priority and Impact were not imputed (section 10).
9. **Checkpoint.** number_cnt and Alert_Status were dropped; the clean file has 46,606 rows and 26 columns (Table 11) and is the input to `04_eda` and `05_split_features`.